## RAG

### Loader

#### langchain_community

项目已不再维护

##### 1、TXT 读取

In [ ]:
from langchain_community.document_loaders import TextLoader
from rich.pretty import pprint

loader = TextLoader(
    file_path="./assets/01-langchain-overview.txt",
    encoding="utf-8",
)

docs = loader.load()
pprint(docs)

##### 2、CSV 读取

In [ ]:
from langchain_community.document_loaders import CSVLoader
from rich.pretty import pprint

loader = CSVLoader(
    file_path="./assets/02-users.csv"
)

docs = loader.load()
pprint(docs)

##### 3、JSON 读取

In [ ]:
from langchain_community.document_loaders import JSONLoader
from rich.pretty import pprint

loader = JSONLoader(
    file_path="./assets/03-knowdage.json",
    # 读取完整对象，并使用 jq 转换为字符串，JSONLoader 转换存在转义问题
    jq_schema=".[] | tojson",
    text_content=False,
)

docs = loader.load()
pprint(docs)


loader2 = JSONLoader(
    file_path="./assets/03-knowdage.json",
    # 读取每个 documents 元素的 summary 字段
    # 还支持很多其它复杂操作
    jq_schema=".documents[].summary",
    text_content=True,
)

docs = loader2.load()
pprint(docs)

##### 4、PDF 读取

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from rich.pretty import pprint

# 能读取文本，但表格、公式和图片等被忽略
loader = PyPDFLoader(
    file_path="./assets/04-rag.pdf",
    extraction_mode="plain"
)

docs = loader.load()
pprint(docs)

### [Unstructured](https://unstructured.io/)

支持丰富的文件格式

In [ ]:
from langchain_unstructured import UnstructuredLoader
from rich.pretty import pprint

loader = UnstructuredLoader(
    file_path="./assets/04-rag.pdf",
    strategy="hi_res",
    languages=["chi_sim", "eng"],
)

docs = loader.load()
pprint(docs)

### [Docling](https://docling.cn/docling/) (推荐)

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, PictureDescriptionVlmEngineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_docling.loader import DoclingLoader, ExportType

options = PdfPipelineOptions(
    do_ocr=True,                  # 识别扫描文字
    do_table_structure=True,      # 重建表格结构
    do_formula_enrichment=True,   # 识别公式，转换为 LaTeX
    do_picture_description=True,  # 使用视觉模型生成图片描述
    picture_description_options=PictureDescriptionVlmEngineOptions.from_preset(
        "qwen",
        picture_area_threshold=0.0,
        prompt="请识别图片中的文字，保留原始语言，不要翻译",
    ),
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=options)
    }
)

loader = DoclingLoader(
    file_path="./assets/04-rag.pdf",
    converter=converter,
    export_type=ExportType.MARKDOWN,
)

documents = loader.load()
print(documents[0].page_content)